# OXIOW — Colab com a ponte para o Kaggle (T4 grátis)

**O que este notebook faz de diferente:** ele não só roda a GPU do Colab — ele **opera a API
do Kaggle dentro do Colab**. Com isso:

- baixa os **modelos que já estão no Kaggle** (11,5 GB do Animate) em vez de re-baixar tudo;
- roda o Animate na **T4 gratuita** do Colab;
- sobe o **resultado** de volta para o Kaggle como dataset;
- e eu puxo o resultado pela CLI, no servidor.

**Antes de rodar:** menu **Runtime ▸ Alterar tipo de ambiente de execução ▸ T4 GPU ▸ Salvar**.

**O que este notebook NÃO resolve:** a cota de GPU do Kaggle (30 h/semana por conta, e não
existe como aumentar). Ele existe para **somar** os dois canais, não para furar a cota.

**Regra da casa:** UMA conta no Kaggle. O Kaggle bane multi-conta para ganhar GPU — declarado
por funcionário deles no fórum oficial. Perder a conta `oxiowhub` custaria os 15+ datasets,
todos os kernels e o acervo. No **Colab** usar duas contas é legítimo.


In [ ]:
# ── 1. AMBIENTE: confere a GPU (tem que ser T4 / sm_75) ──
import os, sys, subprocess, shutil, time, glob, json

def sh(cmd, timeout=7200, quiet=False):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    out = (p.stdout or '') + (p.stderr or '')
    if not quiet:
        print('\n'.join(out.strip().splitlines()[-14:]))
    return p.returncode, out

rc, _ = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader", quiet=True)
print("GPU:", _.strip() or "NENHUMA — ative a T4 em Runtime > Alterar tipo de ambiente")
print(">>> T4/sm_75?", "SIM" if ('T4' in _ or 'sm_75' in _ or '7.5' in _) else "NAO (troque o runtime)")
sh("df -h /content | tail -1")


In [ ]:
# ── 2. SOBE O kaggle.json (a chave da API) ──
# O jeito do tutorial: abre uma caixinha de arquivo. O jeito de producao (sem clique):
# vem do cofre do Colab (Secrets) — assim o notebook roda 100% sozinho.
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)
os.makedirs('/home/ubuntu/.kaggle', exist_ok=True)

caminho = None
try:
    from google.colab import userdata
    kjson = userdata.get('KAGGLE_JSON')          # cole o conteudo do kaggle.json no cofre
    if kjson:
        for d in ('/root/.kaggle', os.path.expanduser('~/.kaggle')):
            os.makedirs(d, exist_ok=True)
            open(os.path.join(d, 'kaggle.json'), 'w').write(kjson)
        caminho = 'cofre do Colab (KAGGLE_JSON)'
except Exception as e:
    print("cofre do Colab indisponivel:", e)

if not caminho:
    print("sem KAGGLE_JSON no cofre — suba o arquivo manualmente:")
    from google.colab import files
    up = files.upload()
    for nome, conteudo in up.items():
        open(os.path.join(os.path.expanduser('~/.kaggle'), 'kaggle.json'), 'wb').write(conteudo)
    caminho = 'upload manual'
print("kaggle.json instalado por:", caminho)

# PERMISSAO 600 — sem isso a CLI do Kaggle recusa. O tutorial nao explica por que.
for d in ('/root/.kaggle', os.path.expanduser('~/.kaggle')):
    p = os.path.join(d, 'kaggle.json')
    if os.path.exists(p):
        os.chmod(p, 0o600)
        print(f"  {p} -> permissao 600 OK")


In [ ]:
# ── 3. INSTALA A CLI DO KAGGLE E CONFERE A IDENTIDADE ──
sh("pip -q install kaggle", timeout=900, quiet=True)
sh("kaggle --version")
sh("kaggle config view")


In [ ]:
# ── 4. A PONTE: baixa os MODELOS do Kaggle (em vez de re-baixar 11,5 GB do HF) ──
# O dataset oxiowhub/oxiow-wan-animate ja tem o Animate Q4, o SAM2, a LoRA e o WORKFLOW.
MODELOS = '/content/kaggle_modelos'
os.makedirs(MODELOS, exist_ok=True)
sh(f"cd {MODELOS} && kaggle datasets download -d oxiowhub/oxiow-wan-animate --unzip", timeout=5400)
sh(f"find {MODELOS} -maxdepth 3 -type f | head -20")
sh(f"du -sh {MODELOS}")


In [ ]:
# ── 5. E A ENTRADA (a ancora da Margaret + o video de movimento) ──
ENTRADA = '/content/entrada'
os.makedirs(ENTRADA, exist_ok=True)
sh(f"cd {ENTRADA} && kaggle datasets download -d oxiowhub/oxiow-animate-entrada --unzip", timeout=900)
sh(f"ls -la {ENTRADA}")


In [ ]:
# ── 6. COMFYUI + os nos do Animate (usa os modelos que vieram do Kaggle) ──
os.chdir('/content')
if not os.path.isdir('/content/ComfyUI'):
    sh("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI", timeout=1200)
sh("pip -q install -r /content/ComfyUI/requirements.txt", timeout=2400)
DONOS = {'ComfyUI-VideoHelperSuite':'Kosinkadink', 'ComfyUI-GGUF':'city96',
         'comfyui_controlnet_aux':'Fannovel16'}
for repo, dono in DONOS.items():
    d = '/content/ComfyUI/custom_nodes/' + repo
    if not os.path.isdir(d):
        sh('git clone --depth 1 https://github.com/' + dono + '/' + repo + '.git ' + d, timeout=900)
        sh('pip -q install -r ' + d + '/requirements.txt', timeout=1200, quiet=True)
# liga os modelos do Kaggle no ComfyUI por LINK (sem copiar 11,5 GB)
for sub in ['diffusion_models','vae','text_encoders','sam2','loras','controlnet_aux']:
    origem = os.path.join(MODELOS, 'models', sub)
    destino = '/content/ComfyUI/models/' + sub
    sh(f"mkdir -p {destino}", quiet=True)
    if os.path.isdir(origem):
        for arq in glob.glob(origem + '/*'):
            sh(f"ln -sfn '{arq}' '{destino}/{os.path.basename(arq)}'", quiet=True)
sh("ls -la /content/ComfyUI/models/diffusion_models/ /content/ComfyUI/models/sam2/ | grep -v '^total'")


In [ ]:
# ── 7. SOBE O COMFYUI ──
import subprocess
proc = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188'],
                        cwd='/content/ComfyUI', stdout=open('/content/comfy.log','w'),
                        stderr=subprocess.STDOUT)
import urllib.request
for i in range(60):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=4)
        print(">>> ComfyUI NO AR em http://127.0.0.1:8188"); break
    except Exception:
        if i % 8 == 7: print('  ... subindo', (i+1)*2, 's')
else:
    print("nao subiu; veja /content/comfy.log")
    print(open('/content/comfy.log').read()[-1500:])


## Como isto se encaixa na arquitetura (sem furar cota, sem arriscar conta)

```
KAGGLE (UMA conta)          guarda os MODELOS · guarda o RESULTADO · orquestra
      ▲                                                    ▲
      │ kaggle datasets download              kaggle datasets create/version
      │                                                    │
COLAB (pode ter 2 contas)   roda a T4 gratuita · interface · resultado
      │
      ▼
EU (servidor)               puxo pela CLI · meço a régua · monto o criativo
```

**A conta do Kaggle é UMA de propósito.** O Kaggle bane multi-conta para ganhar GPU
(declarado por funcionário no fórum oficial). O ganho de +30 h não paga a perda dos 15+
datasets e de todo o histórico. **No Colab, duas contas são legítimas** — cada uma tem a
sua própria cota.
